# 안전모 미착용 탐지 점검

기존에 학습한 best.pt를 사용한다. 이번에는 새로 학습하지 않는다.

원본: https://github.com/jnikem/ppe-missing-detection

수정한 부분
- 다른 클래스의 정답 박스도 확인한다.
- 중복 예측도 FP에 포함한다.
- IoU는 반올림하지 않은 값으로 판정한다.
- 가중치와 결과 폴더를 한 곳에서 지정한다.

학습 가중치는 직접 업로드해야 한다. 이전 노트북의 출력은 옮기지 않았다.


In [ ]:
%pip install ultralytics==8.4.215 pyyaml
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

BASE = Path.cwd() / "ppe_review"
BASE.mkdir(exist_ok=True)


## 1. 가중치 준비
Colab에서는 아래 셀로 best.pt를 올린다. PC에서는 WEIGHTS를 파일 위치로 바꾸면 된다.


In [ ]:
try:
    from google.colab import files
except ImportError:
    WEIGHTS = Path("best.pt").resolve()
else:
    uploaded = files.upload()
    candidates = [Path(p).resolve() for p in uploaded if p.endswith(".pt")]
    if len(candidates) != 1:
        raise ValueError("이번에 사용할 best.pt 한 개만 올려 주세요.")
    WEIGHTS = candidates[0]
assert WEIGHTS.exists(), WEIGHTS
print(WEIGHTS)


## 2. 평가 함수
신뢰도가 높은 예측부터 같은 클래스의 정답과 연결한다. 이것은 고정 신뢰도에서 사례를 세는 별도 기준이다. AP50 계산과 구분해서 사용한다.


In [ ]:
"""안전모 미착용 결과를 같은 기준으로 세기 위한 함수."""

def box_iou(a, b):
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - w*h
    return w*h / union if union else 0.0


def evaluate_image(gt, pred, conf=0.25, iou_limit=0.5):
    # gt: cls, box / pred: cls, box, conf
    # 예측 하나와 정답 하나만 연결한다. 신뢰도가 높은 예측부터 확인한다.
    predictions = [dict(p, pred_id=i) for i, p in enumerate(pred) if p['conf'] >= conf]
    predictions.sort(key=lambda p: (-p['conf'], p['pred_id']))
    used = set()
    pred_rows = []
    for p in predictions:
        same = [(j, box_iou(p['box'], g['box'])) for j, g in enumerate(gt)
                if g['cls'] == p['cls']]
        available = [(j, v) for j, v in same if j not in used]
        best = max(available, key=lambda x: x[1], default=(None, 0.0))
        hit = best[0] is not None and best[1] >= iou_limit
        if hit:
            used.add(best[0])
        nearest = max(same, key=lambda x: x[1], default=(None, 0.0))
        other = [(j, box_iou(p['box'], g['box'])) for j, g in enumerate(gt)
                 if g['cls'] != p['cls']]
        other_best = max(other, key=lambda x: x[1], default=(None, 0.0))
        # IoU만으로 라벨 누락이나 실제 오탐의 원인을 확정하지 않는다.
        if hit:
            kind = 'TP'
        elif nearest[1] >= iou_limit and nearest[0] in used:
            kind = 'FP_duplicate_candidate'
        elif nearest[1] > 0:
            kind = 'FP_overlap_below_threshold'
        elif other_best[1] >= iou_limit:
            kind = 'FP_other_class_overlap'
        else:
            kind = 'FP_no_same_class_overlap'
        pred_rows.append(dict(p, status='TP' if hit else 'FP', kind=kind,
                              gt_id=best[0] if hit else None,
                              max_iou=nearest[1], nearest_gt_id=nearest[0],
                              other_gt_id=other_best[0], other_iou=other_best[1],
                              other_cls=gt[other_best[0]]['cls'] if other_best[0] is not None else None))
    gt_rows = []
    for j, g in enumerate(gt):
        match = next((p for p in pred_rows if p['gt_id'] == j), None)
        different = [(p, box_iou(g['box'], p['box'])) for p in predictions
                     if p['cls'] != g['cls']]
        other, overlap = max(different, key=lambda x: x[1], default=(None, 0.0))
        kind = 'TP' if match else ('FN_other_class_overlap' if overlap >= iou_limit
                                  else 'FN_no_valid_match')
        gt_rows.append(dict(g, gt_id=j, status='TP' if match else 'FN', kind=kind,
                            other_cls=other['cls'] if other and overlap >= iou_limit else None))
    return gt_rows, pred_rows


## 3. 데이터 준비
입력 크기는 기존과 같은 640이다. CPU로도 실행할 수 있다.


In [ ]:
from ultralytics.data.utils import check_det_dataset
URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
d = check_det_dataset(URL)
DATA = Path(d["train"]).parent.parent


## 4. 데이터 확인과 예측
클래스별 개수를 세고, 기존 가중치로 검증 이미지를 예측한다.


In [ ]:
"""기존 가중치로 검증한다. 학습이나 입력 크기 비교는 하지 않는다."""
import argparse
import csv
import hashlib
import json
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw



def write_csv(path, rows):
    if not rows:
        return
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def read_labels(path, width, height):
    rows = []
    if path.exists():
        for line in path.read_text().splitlines():
            cls, cx, cy, bw, bh = map(float, line.split())
            rows.append(dict(cls=int(cls), box=[(cx-bw/2)*width,(cy-bh/2)*height,
                                               (cx+bw/2)*width,(cy+bh/2)*height]))
    return rows

def main():
    from types import SimpleNamespace
    args = SimpleNamespace(data=str(DATA), weights=str(WEIGHTS), audit_only=False)
    root, out = Path(args.data), BASE/'review'
    out.mkdir(exist_ok=True)
    import yaml
    names = yaml.safe_load((root/'construction-ppe.yaml').read_text())['names'] if (root/'construction-ppe.yaml').exists() else None
    if names is None:
        config = next(root.glob('*.yaml'))
        names = yaml.safe_load(config.read_text())['names']
    if isinstance(names,list):
        names = dict(enumerate(names))
    images_by_split = {}
    counts, hashes, errors, bad_labels = [], {}, [], []
    for split in ['train','val']:
        images = sorted(p for p in (root/'images'/split).iterdir() if p.suffix.lower() in ['.jpg','.jpeg','.png'])
        images_by_split[split] = images
        boxes, image_counts = Counter(), Counter()
        label_paths = list((root/'labels'/split).glob('*.txt'))
        for ip in images:
            lp = root/'labels'/split/(ip.stem+'.txt')
            if not lp.exists():
                errors.append(str(lp))
            else:
                for line_no,line in enumerate(lp.read_text().splitlines(),1):
                    fields=line.split()
                    if len(fields)!=5:
                        bad_labels.append(dict(file=lp.name,line=line_no,reason='field_count'))
                        continue
                    c,cx,cy,bw,bh=map(float,fields)
                    if c!=int(c) or int(c) not in names or not (0<=cx<=1 and 0<=cy<=1 and 0<bw<=1 and 0<bh<=1):
                        bad_labels.append(dict(file=lp.name,line=line_no,reason='class_or_coordinate'))
            with Image.open(ip) as im:
                gs = read_labels(lp,*im.size)
            boxes.update(g['cls'] for g in gs)
            image_counts.update(set(g['cls'] for g in gs))
            h = hashlib.sha256(ip.read_bytes()).hexdigest()
            hashes.setdefault(h,[]).append((split,ip.name))
        for cls,name in names.items():
            counts.append(dict(split=split,cls=cls,name=name,images=image_counts[cls],boxes=boxes[cls]))
        print(split,len(images),'images',len(label_paths),'labels',flush=True)
    duplicates = [v for v in hashes.values() if len({x[0] for x in v})>1]
    write_csv(out/'class_counts.csv',counts)
    audit = dict(images={s:len(v) for s,v in images_by_split.items()},missing_labels=errors,
                 bad_label_rows=bad_labels,
                 exact_cross_split_duplicates=duplicates, note='파일 내용이 같은 중복만 확인. 비슷한 장면은 별도 검토 필요.')
    (out/'data_audit.json').write_text(json.dumps(audit,indent=2,ensure_ascii=False),encoding='utf-8')
    if args.audit_only:
        return
    from ultralytics import YOLO
    import ultralytics, torch
    model = YOLO(args.weights)
    if model.names != names:
        raise ValueError('데이터와 모델의 클래스 번호가 다릅니다.')
    images = images_by_split['val']
    gt_rows, pred_rows, raw = [], [], []
    for i,r in enumerate(model.predict([str(p) for p in images],imgsz=640,conf=0.25,
                                       device='cpu',stream=True,verbose=False)):
        ip = Path(r.path)
        with Image.open(ip) as im:
            width,height = im.size
        gt = read_labels(root/'labels/val'/(ip.stem+'.txt'),width,height)
        pred = [dict(cls=int(c),conf=float(s),box=b) for c,s,b in zip(
            r.boxes.cls.cpu().tolist(),r.boxes.conf.cpu().tolist(),r.boxes.xyxy.cpu().tolist())]
        gs, ps = evaluate_image(gt,pred)
        raw.append(dict(img=ip.name,width=width,height=height,gt=gt,pred=pred))
        gt_rows.extend(dict(img=ip.name,**g) for g in gs)
        pred_rows.extend(dict(img=ip.name,**p) for p in ps)
        if (i+1)%20 == 0:
            print('predict',i+1,'/',len(images),flush=True)
    (out/'predictions.json').write_text(json.dumps(raw),encoding='utf-8')
    write_csv(out/'gt_review.csv',gt_rows)
    write_csv(out/'pred_review.csv',pred_rows)
    summary=[]
    for cls,name in names.items():
        gs=[g for g in gt_rows if g['cls']==cls]
        ps=[p for p in pred_rows if p['cls']==cls]
        tp=sum(p['status']=='TP' for p in ps)
        fp=len(ps)-tp
        fn=len(gs)-tp
        summary.append(dict(cls=cls,name=name,GT=len(gs),TP=tp,FP=fp,FN=fn,
                            precision=tp/(tp+fp) if tp+fp else None,
                            recall=tp/len(gs) if gs else None))
    write_csv(out/'fixed_threshold_metrics.csv',summary)
    (out/'run_settings.json').write_text(json.dumps(dict(
        ultralytics=ultralytics.__version__,torch=torch.__version__,device='cpu',imgsz=640,
        conf=0.25,match_iou=0.5,matching='confidence descending, best unused same-class GT',
        weights_sha256=hashlib.sha256(Path(args.weights).read_bytes()).hexdigest()),indent=2),encoding='utf-8')
    print(json.dumps(summary,ensure_ascii=False),flush=True)

main()

display(pd.read_csv(BASE/"review/class_counts.csv"))
display(pd.read_csv(BASE/"review/fixed_threshold_metrics.csv"))


## 5. 오탐 확인
no_helmet 정답과 안 겹쳐도 helmet 정답과 겹칠 수 있다. 자동 분류는 후보이며 원인 확정은 이미지 확인 후에 한다.


In [ ]:
pred = pd.read_csv(BASE/"review/pred_review.csv")
fp = pred[(pred["cls"] == 6) & (pred["status"] == "FP")]
display(fp[["img", "conf", "kind", "max_iou", "other_cls", "other_iou"]])
print(fp["kind"].value_counts())


## 6. 이미지 한 장 시연
검증 이미지 중 하나를 표시한다. 다른 이미지를 쓰려면 SAMPLE을 바꾸면 된다.


In [ ]:
from ultralytics import YOLO
model = YOLO(str(WEIGHTS))
SAMPLE = sorted(Path(d["val"]).glob("*.jpg"))[0]
result = model.predict(str(SAMPLE), imgsz=640, conf=0.25, device="cpu", verbose=False)[0]
display(__import__("PIL.Image",fromlist=["Image"]).fromarray(result.plot()[..., ::-1]))
